
# Activity 2: Text Classification with LSTM

## Task 1: Reuse the Dataset, Train/Test Split, and Tokenization


In [2]:

import pandas as pd
import torch
import torch.nn as nn
import tiktoken

from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset, DataLoader

# Load the same dataset and select the same sample
df = pd.read_csv("D:\Projects\GenAI\IMDB Dataset.csv")
df = df.sample(n=2000, random_state=42).reset_index(drop=True)

# Convert sentiment labels into numbers
df["label"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

# Split the dataset exactly as in Exercise 1
X = df["review"]
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Tokenize using cl100k_base
encoding = tiktoken.get_encoding("cl100k_base")

X_train_tokens = [
    encoding.encode(review, disallowed_special=())
    for review in X_train
]
X_test_tokens = [
    encoding.encode(review, disallowed_special=())
    for review in X_test
]

print("Training reviews:", len(X_train_tokens))
print("Testing reviews:", len(X_test_tokens))
print("Tokenization:", encoding.name)


<>:10: SyntaxWarning: invalid escape sequence '\P'
<>:10: SyntaxWarning: invalid escape sequence '\P'
C:\Users\prana\AppData\Local\Temp\ipykernel_17876\3792922575.py:10: SyntaxWarning: invalid escape sequence '\P'
  df = pd.read_csv("D:\Projects\GenAI\IMDB Dataset.csv")


Training reviews: 1600
Testing reviews: 400
Tokenization: cl100k_base



## Task 2: Pad/Truncate Sequences and Create Tensors


In [3]:

# Set the same fixed sequence length as Exercise 1
MAX_LENGTH = 200

# Pad or truncate token sequences
def pad_or_truncate(token_sequences, max_length):
    padded_sequences = []

    for tokens in token_sequences:
        tokens = tokens[:max_length]
        tokens = tokens + [0] * (max_length - len(tokens))
        padded_sequences.append(tokens)

    return padded_sequences

X_train_padded = pad_or_truncate(X_train_tokens, MAX_LENGTH)
X_test_padded = pad_or_truncate(X_test_tokens, MAX_LENGTH)

# Convert inputs into PyTorch tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

# Convert labels into PyTorch tensors
y_train_tensor = torch.tensor(y_train.to_numpy(), dtype=torch.long)
y_test_tensor = torch.tensor(y_test.to_numpy(), dtype=torch.long)

# Verify shapes
print("Training input shape:", X_train_tensor.shape)
print("Testing input shape:", X_test_tensor.shape)
print("Training labels shape:", y_train_tensor.shape)
print("Testing labels shape:", y_test_tensor.shape)


Training input shape: torch.Size([1600, 200])
Testing input shape: torch.Size([400, 200])
Training labels shape: torch.Size([1600])
Testing labels shape: torch.Size([400])


Task 3 : Build the LSTM Model

In [4]:

class LSTMClassifier(nn.Module):
    def __init__(self, vocab_size, embedding_dim=64, hidden_dim=64):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embedding_dim)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            batch_first=True
        )
        self.fc = nn.Linear(hidden_dim, 2)

    def forward(self, x):
        embedded = self.embedding(x)
        output, (hidden, cell) = self.lstm(embedded)
        last_hidden = hidden[-1]
        return self.fc(last_hidden)


vocab_size = encoding.n_vocab
model = LSTMClassifier(vocab_size)

print(model)


LSTMClassifier(
  (embedding): Embedding(100277, 64)
  (lstm): LSTM(64, 64, batch_first=True)
  (fc): Linear(in_features=64, out_features=2, bias=True)
)


Task 4: Train the LSTM Model

In [5]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

num_epochs = 5

for epoch in range(num_epochs):
    model.train()
    total_loss = 0

    for inputs, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    average_loss = total_loss / len(train_loader)
    print(f"Epoch [{epoch + 1}/{num_epochs}], Loss: {average_loss:.4f}")


Epoch [1/5], Loss: 0.6976
Epoch [2/5], Loss: 0.6885
Epoch [3/5], Loss: 0.6762
Epoch [4/5], Loss: 0.6440
Epoch [5/5], Loss: 0.5814


Task 5: Test the LSTM Model and Calculate Accuracy

In [6]:

model.eval()

with torch.no_grad():
    outputs = model(X_test_tensor)
    predictions = torch.argmax(outputs, dim=1)

    correct = (predictions == y_test_tensor).sum().item()
    total = y_test_tensor.size(0)
    accuracy = correct / total * 100

print(f"Correct Predictions: {correct}/{total}")
print(f"LSTM Test Accuracy: {accuracy:.2f}%")


Correct Predictions: 213/400
LSTM Test Accuracy: 53.25%


Task 6: Test Custom Movie Reviews

In [7]:

custom_reviews = [
    "This movie was fantastic and I really enjoyed it.",
    "The film was boring and the acting was terrible.",
    "The story was interesting, but the ending was disappointing."
]

model.eval()

for review in custom_reviews:
    tokens = encoding.encode(review, disallowed_special=())
    tokens = tokens[:MAX_LENGTH]
    tokens += [0] * (MAX_LENGTH - len(tokens))

    input_tensor = torch.tensor([tokens], dtype=torch.long)

    with torch.no_grad():
        output = model(input_tensor)
        prediction = torch.argmax(output, dim=1).item()

    sentiment = "Positive" if prediction == 1 else "Negative"
    print("Review:", review)
    print("Predicted Sentiment:", sentiment)
    print()


Review: This movie was fantastic and I really enjoyed it.
Predicted Sentiment: Negative

Review: The film was boring and the acting was terrible.
Predicted Sentiment: Negative

Review: The story was interesting, but the ending was disappointing.
Predicted Sentiment: Negative



Task 7: Compare RNN and LSTM Results


## Task 7: Comparison of RNN and LSTM

| Parameter | RNN | LSTM |
|---|---|---|
| Test Accuracy | 50.75% | 53.25% |
| Correct Predictions | 203/400 | 213/400 |
| Epochs | 5 | 5 |
| Model | nn.RNN | nn.LSTM |

### Guiding Questions

**1. Which model achieved higher accuracy?**

The LSTM model achieved higher accuracy (53.25%) than the RNN model (50.75%).

**2. Did LSTM reach lower training loss or need fewer epochs?**

Both models were trained for 5 epochs. The RNN training loss decreased from 0.7038 to 0.5861. The LSTM training loss should be compared using its recorded epoch values to determine whether it reached a lower loss.

**3. Did the custom reviews agree with the expected sentiment?**

The LSTM predicted all three custom reviews as Negative. Therefore, it correctly classified the negative review but incorrectly classified the positive review. The third review has mixed sentiment, so its expected label is less clear.

**4. Why can LSTM perform differently from RNN?**

A standard RNN can struggle to retain information over long sequences. LSTM uses a cell state and gates to control what information is remembered, forgotten, and passed forward. This can help it learn longer-term relationships in text.
